<a href="https://colab.research.google.com/github/Z-9000/ML_NN_DL/blob/main/Micrograd_implementation/Micrograd_implementation_copy_in_google_collab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class Value:

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"    # giving an assertion error in case the other is neither int nor float
        out = Value(self.data**other, (self,), f'**{other}')

        def _backward():
            self.grad += other * (self.data ** (other -1)) * out.grad
        out._backward = _backward

        return out

    def __rmul__(self, other):  # for the case when we are multiplying a number to a Value node instead of a Value node to a number
        return self * other

    def __truediv__(self, other):
        return self * other**-1

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __radd__(self, other):  # similar reasoning to rmul
        return self + other

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')      # creating the value node of the output

        def _backward():                     # defining the backward function for the tanh to call during the back propogation
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward    # the backward function is stored in the new output value of that is created from the operation

        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward

        return out


    def backward(self):       # defining the backward function that the user will call

        topo = []
        visited = set()      # a set to keep track of the nodes that have been visited

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)   # calling the build_topo function on the child nodes to recursively add all the nodes in the visited set
                topo.append(v)
        build_topo(self)

        self.grad = 1.0    # setting the self.grad to 1 since the grad of the thing to itself is 1
        for node in reversed(topo):    # iterating through the set from the last node to the first node
            node._backward()   # calling the _backward function that is defined for the particular node

In [ ]:
# defining inputs for the neuron
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

# defining weights for the neurons inputs
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

# defining bias for the neuron
b = Value(6.8813735870195432, label='b')

# doing the standard multiplication of the inputs with the weights and adding them all and later adding the biases
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'   # taking the tanh of the output to add non linearity to the output and squashing the values of the nn into a predictable clean range

In [ ]:
# defining inputs for the neuron
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

# defining weights for the neurons inputs
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

# defining bias for the neuron
b = Value(6.8813735870195432, label='b')

# doing the standard multiplication of the inputs with the weights and adding them all and later adding the biases
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'

e = (2*n).exp()
o = (e - 1)/(e + 1)

o.label = 'o'
o.backward()

In [ ]:
class Neuron:

    def __init__(self, nin):    # defining random random numbers i.e. weights and biases (both between -1 and 1) for the nin number of inputs each
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        # w * x + b
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()   # activation function is called on each of the neuron in the list
        return out    # whenever we call the Neuron object with the inputs x(list) then this will be called and the value will be returned

    def parameters(self):
        return self.w + [self.b]

class Layer:

    def __init__(self, nin, nout):   # this will create the layer of nout number of neurons for the nin number of input neurons where each of the nin neurons is connected to the every single nout number of neurons of layer and all these nout number of neurons will be added to the list "neurons"
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]   # activation function is called for each of the neuron in each of the list of neurons in the neurons list
        return outs[0] if len(outs) == 1 else outs   # for making the output look cleaner in case only 1 output is there

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:

    def __init__(self, nin, nout):
        sz = [nin] + nout
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nout))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [ ]:
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets

In [ ]:
for k in range(20):

  # forward pass
  ypred = [n(x) for x in xs]
  loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))

  # backward pass
  for p in n.parameters():
    p.grad = 0.0
  loss.backward()

  # update
  for p in n.parameters():
    p.data += -0.1 * p.grad

  print(k, loss.data)

0 0.00147210010251298
1 0.0014637044728384456
2 0.001455402078751109
3 0.0014471913880942044
4 0.0014390709019725353
5 0.0014310391538562227
6 0.0014230947087134644
7 0.0014152361621707825
8 0.0014074621397001396
9 0.0013997712958317148
10 0.0013921623133914317
11 0.0013846339027624261
12 0.0013771848011694052
13 0.0013698137719852811
14 0.0013625196040590935
15 0.0013553011110646318
16 0.0013481571308687685
17 0.0013410865249191858
18 0.0013340881776504065
19 0.0013271609959077423


In [ ]:
ypred

[Value(data=0.9832997089936077),
 Value(data=-0.9894663048437116),
 Value(data=-0.9775077427315214),
 Value(data=0.9792298072822491)]